In the quick start we'll show you know to build a simple LLM application with LangChain.This application will translate text from English into another- it's just a single LLM call plus some prompting. Still this is a great way to get started with LangChain - a lot feature can be built with just some prompting and LLM call!

after seeing this video , you'll have a high level overview of:
<li>Using Language models</li>
<li>Using Prompt Templates and OutputParsers</li>
<li>Using Langchain Expression Language(LCEL) to chain components  together</li>
<li>Debugging and tracing your application using LangSmith </li>
<li>Deploying your application with LangServe</li>

In this section we'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.
Note that this chat that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:
<li>Conversational RAG: Enable a chatbot experience over an external source of data</li>
<li>Agents: Build a chatbot that can take actions</li>


In [21]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [22]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "Qwen/Qwen3.6-27B" , groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '0.3.27'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001CF0E3D0070>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CF0E3D0040>, model_name='Qwen/Qwen3.6-27B', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [23]:
from langchain_core.messages import HumanMessage
result = model.invoke([HumanMessage(content="Hi , my name is Muskan Chauhan and AI Forward Deployed Engineer")])
result

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Name:** Muskan Chauhan\n   - **Role/Title:** AI Forward Deployed Engineer\n   - **Greeting:** "Hi , my name is..."\n\n2.  **Identify Key Elements:**\n   - Personal introduction\n   - Professional title (AI Forward Deployed Engineer - likely a role at Microsoft or similar tech company, focusing on deploying AI solutions with customers)\n   - Open-ended, expects a friendly/professional response\n\n3.  **Determine Response Goals:**\n   - Acknowledge and greet by name\n   - Acknowledge the professional role\n   - Express enthusiasm/offer assistance\n   - Keep it professional, concise, and open-ended\n   - Match tone: friendly, professional, tech-aware\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Muskan! It\'s great to meet you. As an AI Forward Deployed Engineer, you\'re right at the forefront of bringing AI solutions to life for customers. How can I assist you today? Whether it\'s bra

In [24]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Name:** Muskan Chauhan\n   - **Role/Title:** AI Forward Deployed Engineer\n   - **Greeting:** "Hi , my name is..."\n\n2.  **Identify Key Elements:**\n   - Personal introduction\n   - Professional title (AI Forward Deployed Engineer - likely a role at Microsoft or similar tech company, focusing on deploying AI solutions with customers)\n   - Open-ended, expects a friendly/professional response\n\n3.  **Determine Response Goals:**\n   - Acknowledge and greet by name\n   - Acknowledge the professional role\n   - Express enthusiasm/offer assistance\n   - Keep it professional, concise, and open-ended\n   - Match tone: friendly, professional, tech-aware\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Muskan! It\'s great to meet you. As an AI Forward Deployed Engineer, you\'re right at the forefront of bringing AI solutions to life for customers. How can I assist you today? Whether it\'s brainstorming impleme

In [25]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi , My name is Muskan Chauhan "),
    AIMessage(content="Hi Muskan! Nice to meet you. As an AI Forward Deployed Engineer, you\'re clearly working at the exciting intersection of AI development and real-world customer impact. How can I assist you today? Whether it\'s debugging ML pipelines, optimizing model deployments, exploring new AI frameworks, or brainstorming solutions for enterprise AI use cases, I\'m here to help."),
    HumanMessage(content="What kind of salary i am expected in FDE role")
])

 


AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User: "What kind of salary i am expected in FDE role"\n   - Context: User previously identified as "Muskan Chauhan" and I assumed an "AI Forward Deployed Engineer" role based on my opening message (which was a bit presumptuous, but she didn\'t correct it). Now she\'s asking about salary expectations for an FDE role.\n   - Key terms: FDE role, salary expectations\n   - Need to clarify: FDE can stand for different things in tech:\n     - Forward Deployed Engineer (Google, Microsoft, etc.)\n     - Full-Stack Developer/Engineer (sometimes abbreviated, but less common)\n     - Field Deployment Engineer\n     - Financial Data Engineer\n     - I should assume Forward Deployed Engineer since it\'s a well-known role at major tech companies, but I should also acknowledge the ambiguity and ask for clarification if needed.\n\n   - Key factors affecting salary:\n     - Company (Google, Microsoft, Meta, st

### Message History
We can use a Message Hsitory class to wrap our model and male it stateful.This will Keep track of inputs and outputs of the model and store them in some datastore. Future Interaction will then load those message and pass them into chain as pat of the input , Let's see how to use this !

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}


def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model , get_session_history)


e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [27]:
config = {"configurable":{"session_id":"chat1"}}

In [28]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer")
    ],
    config=config
)

In [29]:
## Chnage the config  --> session id
config1 = {"configurable":{"session_id":"chat1"}}
response = with_message_history.invoke(
    [
        HumanMessage(content="Whats my name")
    ],
    config=config1
)
response.content


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asked "Whats my name"\n2.  **Check Context:** In the first message, the user explicitly stated: "Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer"\n3.  **Extract Information:** Name = Muskan Chauhan\n4.  **Formulate Response:** Acknowledge the name clearly and politely. Keep it concise.\n5.  **Self-Correction/Verification:** Double-check the spelling from the prompt: "Muskan Chauhan". Match exactly. No extra fluff needed. Ready.✅\n</think>\n\nYour name is **Muskan Chauhan**. Nice to meet you! How can I help you today? 😊'

In [30]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer")
    ],
    config=config
)

### Prompt Templates 
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case , the raw user input in just a message , which we are passing to the LLM. Let's now make that a bit more complicated. FFisrt , let's add in a system messsagewith some custom instructions (But still taking message as input.) Next , we'll add in more input besides just the message.

In [31]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the nest of your ability."),
        MessagesPlaceholder(variable_name="message")
    ]

)

chain = prompt|model

In [32]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [33]:
config = {
    "configurable": {
        "session_id": "chat1"
    }
}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asked "What's my name?"
2.  **Check Context:** In previous turns, the user explicitly stated: "Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer"
3.  **Formulate Response:** Directly answer the question based on the provided context. Keep it polite and concise.
4.  **Draft Response:** Your name is Muskan Chauhan.
5.  **Refine:** Add a friendly touch if appropriate, but keep it direct. "Your name is Muskan Chauhan." matches the prompt's simplicity. I'll stick to that.✅
</think>

Your name is **Muskan Chauhan**. Let me know how I can help you today! 😊


In [34]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistent.Answer All question to the best of your ability is {language}"
        ),
        MessagesPlaceholder(variable_name='message')
    ]
)

chain = prompt | model

In [35]:
response = chain.invoke({"message": [HumanMessage(content="Hi My name is Muskan Chauhan")],
                        "language":"Hindi"})
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi My name is Muskan Chauhan"
   - Language: English
   - Intent: Greeting and introducing themselves

2.  **Identify Constraints:**
   - System prompt: "Answer All question to the best of your ability is Hindi"
   - This means I must respond in Hindi, regardless of the input language.

3.  **Formulate Response (Internal Translation & Adaptation):**
   - Acknowledge the greeting and name
   - Respond warmly in Hindi
   - Offer assistance
   - Keep it natural and polite

   Draft in Hindi:
   "नमस्ते मुस्कन चौहान जी! आपका स्वागत है। मैं आपकी कैसे सहायता कर सकता हूँ?"

4.  **Check Against Constraints:**
   - Is it in Hindi? Yes.
   - Does it acknowledge the name? Yes.
   - Does it offer help? Yes.
   - Tone appropriate? Yes, polite and friendly.

5.  **Final Output Generation:** (Matches the draft)
   "नमस्ते मुस्कन चौहान जी! आपका स्वागत है। मैं आपकी कैसे सहायता कर सकता हूँ?"✅
   All constraints met. Ready.

Let's now wrap this more complicated chain in a Message Histiry class. This time, becuase there are multiple keys in the input , we need to specify the correct key to use to save the chat history 

In [36]:
with_message_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="message"
)

e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [37]:
config = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    {"message":[HumanMessage(content="Hi I am Muskan")],
     "language":"Hindi"},
     config=config
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi I am Muskan"
   - Language: English
   - My system prompt: "Answer All question to the best of your ability is Hindi" (Note: The phrasing is slightly awkward, but it clearly means I should respond in Hindi to all questions/prompts)

2.  **Identify Key Requirements:**
   - Acknowledge the user's name (Muskan)
   - Respond in Hindi as instructed
   - Keep it friendly and helpful

3.  **Formulate Response (Internal Translation/Construction):**
   - English: "Hello Muskan! Nice to meet you. How can I help you today?"
   - Hindi: "नमस्ते मुस्कन! आपसे मिलकर अच्छा लगा। आज मैं आपकी कैसे मदद कर सकता हूँ?"
   - Check tone: Friendly, polite, matches the instruction.
   - Check language: Pure Hindi, natural phrasing.

4.  **Final Output Generation:** (matches the Hindi version)
   - "नमस्ते मुस्कन! आपसे मिलकर बहुत अच्छा लगा। आज मैं आपकी कैसे मदद कर सकता हूँ?" (Slightly refined for natural flow)

   All steps align

## Managing the Conversation History

One import concept to unerstand when building chatsbot is how to manage conversation history. if left unmanaged the list of message will grow unbounded and potentially overflow the context windows of the LLM.. Therefore it is important to add a step that limits the size of the messages you are passing in. trim_message helper to reduces how many message we're sending to the model. the timmer allows us  to specify how many tokens we want to keeps, along with other aparmeters like if we want to always keep the system message and whenther to allow partial message.

In [40]:
from langchain_core.messages import SystemMessage , trim_messages
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

message = [
    SystemMessage(content="you're a good assitant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!")
]

trimmer.invoke(message)

e:\GenerativeAi_python\venv\lib\site-packages\langchain_core\language_models\base.py:447: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

e:\GenerativeAi_python\venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Bhuvan Chauhan\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[SystemMessage(content="you're a good assitant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={

In [42]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain=(
    RunnablePassthrough.assign(message=itemgetter("message")|trimmer)
    |prompt
    |model
)

response = chain.invoke(
    {
  "message":message + [HumanMessage(content="what ice cream do i like")],
    "language":"English"
}
)

print(response.content)



<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "what ice cream do i like"
2.  **Scan Conversation History:** Look for previous mentions of ice cream preferences.
   - User said: "I like vanilla ice cream"
3.  **Formulate Response:** Based on the history, the user likes vanilla ice cream. Keep it concise and friendly as per the established tone.
   - Response: "You said you like vanilla ice cream!" or "Vanilla ice cream!"
4.  **Check Constraints:** The system prompt says "Answer All question to the best of your ability is English". The tone so far has been very brief. I'll keep it consistent but clear.
   - "Vanilla ice cream!" fits well.
5.  **Output Generation:** "Vanilla ice cream!" (or similar concise answer)✅
</think>

Vanilla ice cream!


In [43]:
## Lets wrap this in the message History

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="message"
)

config = {"configurable":{"session_id":"chat3"}}

e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [44]:
response = with_message_history.invoke(
    {
        "message":message + [HumanMessage(content= "whats my name?")],
        "language":"English"
    },
    config= config,
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "whats my name?"
2.  **Check Context/History:** Looking back at the conversation history:
   - User: "hi! I'm bob"
   - Assistant: "hi!"
   - User: "I like vanilla ice cream"
   - Assistant: "nice"
   - User: "whats 2 + 2"
   - Assistant: "4"
   - User: "thanks"
   - Assistant: "no problem"
   - User: "having fun?"
   - Assistant: "yes!"
   - User: "whats my name?"
3.  **Identify Key Information:** The user explicitly stated their name is "bob" in the first message.
4.  **Formulate Response:** Acknowledge the name clearly and politely. Keep it consistent with the conversational tone. "Your name is Bob!" or "You told me your name is Bob!"
5.  **Check Constraints:** The prompt says "Answer All question to the best of your ability is English you're a good assitant". (Note: slight typo in prompt "is English" -> "in English", but the intent is clear). Keep it in English.
6.  **Draft Response:** Your name is Bob! 